# A/H Detection Pipeline
Binary classification (With A-H vs No A-H) on BAH dataset frames using a causal transformer.

**Pipeline:**
1. Dataset statistics (split-frames + SegmentedFrames)
2. PyTorch Dataset / DataLoader
3. Causal Transformer model
4. Training (1 epoch)
5. Inference
6. Classification metrics

In [ ]:
import os
import math
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
)

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Paths ────────────────────────────────────────────────────────────────
DATA_DIR         = Path('data')
SPLIT_FRAMES_DIR = DATA_DIR / 'split-frames'
FRAMES_DIR       = DATA_DIR / 'Frames'
SEG_FRAMES_DIR   = DATA_DIR / 'SegmentedFrames'

TRAIN_TXT = SPLIT_FRAMES_DIR / 'train.txt'
TEST_TXT  = SPLIT_FRAMES_DIR / 'test.txt'

# ── Device ───────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'DATA_DIR         : {DATA_DIR.resolve()}')
print(f'FRAMES_DIR       : {FRAMES_DIR}  | exists: {FRAMES_DIR.exists()}')
print(f'SEG_FRAMES_DIR   : {SEG_FRAMES_DIR}  | exists: {SEG_FRAMES_DIR.exists()}')
print(f'TRAIN_TXT        : {TRAIN_TXT}  | exists: {TRAIN_TXT.exists()}')
print(f'TEST_TXT         : {TEST_TXT}  | exists: {TEST_TXT.exists()}')
print(f'Device           : {DEVICE}')

## 1. Dataset Statistics

In [ ]:
def load_split_file(path: Path) -> pd.DataFrame:
    """Read a split-frames .txt file.
    Each line: Videos/PATIENT/Visite_N/VIDEO.mp4/frame-N.jpg,LABEL
    """
    records = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.rsplit(',', 1)
            if len(parts) != 2:
                continue
            frame_rel, label_str = parts
            records.append({'frame_path': frame_rel, 'label': int(label_str)})
    return pd.DataFrame(records)


print('Loading split files...')
train_df = load_split_file(TRAIN_TXT)
test_df  = load_split_file(TEST_TXT)

CLASS_NAMES = {0: 'No A-H', 1: 'With A-H'}


def split_stats(df: pd.DataFrame, name: str) -> None:
    total  = len(df)
    counts = df['label'].value_counts().sort_index()
    print(f'\n── {name} ──')
    print(f'  Total frames : {total:,}')
    for cls_id, cnt in counts.items():
        print(f'  Class {cls_id} ({CLASS_NAMES[cls_id]:>8s}): {cnt:>8,}  ({cnt/total*100:.1f}%)')


split_stats(train_df, 'TRAIN')
split_stats(test_df,  'TEST')

# ── Segmented frames coverage ─────────────────────────────────────────────
print('\n── SegmentedFrames coverage ──')
if SEG_FRAMES_DIR.exists():
    seg_files = list(SEG_FRAMES_DIR.rglob('*.jpg'))
    print(f'  Total segmented files found: {len(seg_files):,}')
    sample_size = min(5000, len(train_df))
    sample_df   = train_df.sample(sample_size, random_state=SEED)
    covered = 0
    for fp in sample_df['frame_path']:
        stem   = Path(fp).stem
        parent = str(Path(fp).parent)
        seg_dir = SEG_FRAMES_DIR / parent
        if seg_dir.exists() and list(seg_dir.glob(f'{stem}_person-*.jpg')):
            covered += 1
    print(f'  Train sample ({sample_size:,} frames): {covered:,} have segmented version '
          f'({covered/sample_size*100:.1f}%)')
else:
    print(f'  SegmentedFrames directory not found: {SEG_FRAMES_DIR}')
    print('  Will use original Frames only.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (df, title) in zip(axes, [(train_df, 'Train'), (test_df, 'Test')]):
    counts = df['label'].value_counts().sort_index()
    bars = ax.bar([CLASS_NAMES[i] for i in counts.index], counts.values,
                  color=['#4C72B0', '#DD8452'])
    ax.set_title(f'{title} split  (n={len(df):,})', fontsize=12)
    ax.set_ylabel('Frame count')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
                f'{val:,}', ha='center', va='bottom', fontsize=9)
plt.suptitle('Class Distribution in BAH Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. PyTorch Dataset & DataLoader

In [ ]:
# ── Hyper-parameters ─────────────────────────────────────────────────────
IMG_SIZE    = 224    # resize all frames to IMG_SIZE x IMG_SIZE
PATCH_SIZE  = 16     # transformer patch size
EMBED_DIM   = 256    # transformer embedding dimension
NUM_HEADS   = 8      # attention heads
NUM_LAYERS  = 4      # transformer encoder layers
MLP_RATIO   = 4.0    # MLP hidden dim = EMBED_DIM * MLP_RATIO
DROPOUT     = 0.1
NUM_CLASSES = 2

BATCH_SIZE  = 32
LR          = 3e-4
NUM_EPOCHS  = 1

# Subsample large splits to keep 1-epoch demo tractable (set None for full data)
MAX_TRAIN_SAMPLES = 20_000
MAX_TEST_SAMPLES  = 5_000

print('Hyper-parameters:')
print(f'  IMG_SIZE={IMG_SIZE}, PATCH_SIZE={PATCH_SIZE}, EMBED_DIM={EMBED_DIM}')
print(f'  NUM_HEADS={NUM_HEADS}, NUM_LAYERS={NUM_LAYERS}')
print(f'  BATCH_SIZE={BATCH_SIZE}, LR={LR}, NUM_EPOCHS={NUM_EPOCHS}')
print(f'  MAX_TRAIN_SAMPLES={MAX_TRAIN_SAMPLES}, MAX_TEST_SAMPLES={MAX_TEST_SAMPLES}')

In [ ]:
class AHFrameDataset(Dataset):
    """Dataset for A/H frame-level binary classification.

    Tries to load the segmented version of each frame first;
    falls back to the original frame if segmentation is unavailable.
    When multiple segmented persons exist for one frame, the first one
    (sorted by person id) is used.
    """

    def __init__(
        self,
        df: pd.DataFrame,
        frames_dir: Path,
        seg_frames_dir: Path,
        transform=None,
        use_segmented: bool = True,
        max_samples: int = None,
    ):
        if max_samples is not None and max_samples < len(df):
            df = (
                df.groupby('label', group_keys=False)
                  .apply(lambda g: g.sample(
                      min(len(g), max_samples // df['label'].nunique()),
                      random_state=SEED))
                  .reset_index(drop=True)
            )
        self.records       = df.reset_index(drop=True)
        self.frames_dir    = frames_dir
        self.seg_dir       = seg_frames_dir
        self.use_segmented = use_segmented
        self.transform     = transform

    def __len__(self) -> int:
        return len(self.records)

    def _resolve_path(self, frame_rel: str) -> Path:
        if self.use_segmented and self.seg_dir.exists():
            stem   = Path(frame_rel).stem
            parent = str(Path(frame_rel).parent)
            seg_folder = self.seg_dir / parent
            candidates = sorted(seg_folder.glob(f'{stem}_person-*.jpg'))
            if candidates:
                return candidates[0]
        return self.frames_dir / frame_rel

    def __getitem__(self, idx: int):
        row      = self.records.iloc[idx]
        label    = int(row['label'])
        img_path = self._resolve_path(row['frame_path'])
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), color=0)
        if self.transform:
            img = self.transform(img)
        return img, label


# ── Transforms ───────────────────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

test_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ── Datasets ─────────────────────────────────────────────────────────────
train_dataset = AHFrameDataset(
    train_df, FRAMES_DIR, SEG_FRAMES_DIR,
    transform=train_transform,
    use_segmented=SEG_FRAMES_DIR.exists(),
    max_samples=MAX_TRAIN_SAMPLES,
)
test_dataset = AHFrameDataset(
    test_df, FRAMES_DIR, SEG_FRAMES_DIR,
    transform=test_transform,
    use_segmented=SEG_FRAMES_DIR.exists(),
    max_samples=MAX_TEST_SAMPLES,
)

print(f'Train dataset size : {len(train_dataset):,}')
print(f'Test  dataset size : {len(test_dataset):,}')

# ── DataLoaders ───────────────────────────────────────────────────────────
NUM_WORKERS = min(4, os.cpu_count() or 1)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE != 'cpu'),
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE != 'cpu'),
)

print(f'Train batches : {len(train_loader):,}')
print(f'Test  batches : {len(test_loader):,}')

In [ ]:
def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    t = tensor.clone()
    for c, (m, s) in enumerate(zip(mean, std)):
        t[c] = t[c] * s + m
    return t.clamp(0, 1)

imgs, labels = next(iter(train_loader))
n_show = min(16, len(imgs))
cols   = 8
rows   = math.ceil(n_show / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.5))
axes = np.array(axes).reshape(rows, cols)
for i, ax in enumerate(axes.flat):
    if i < n_show:
        img_np = denormalize(imgs[i]).permute(1, 2, 0).numpy()
        ax.imshow(img_np)
        ax.set_title(CLASS_NAMES[labels[i].item()], fontsize=7)
    ax.axis('off')
plt.suptitle('Sample training batch', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Causal Transformer Model

Architecture:
1. **Patch Embedding** — split image into non-overlapping patches, project to `EMBED_DIM`
2. **[CLS] token** — prepended learnable classification token
3. **Positional Embedding** — learnable 1-D positional encodings
4. **Causal Transformer Encoder** — multi-head self-attention with a **causal (lower-triangular) mask** so each patch only attends to itself and earlier patches (left-to-right raster order)
5. **Classification Head** — MLP on the [CLS] token output → 2 logits

In [ ]:
class PatchEmbedding(nn.Module):
    """Split image into patches and project to embedding space."""

    def __init__(self, img_size: int, patch_size: int, in_channels: int, embed_dim: int):
        super().__init__()
        assert img_size % patch_size == 0, 'img_size must be divisible by patch_size'
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.proj(x)       # (B, D, H/P, W/P)
        x = x.flatten(2)       # (B, D, N)
        x = x.transpose(1, 2)  # (B, N, D)
        return x


class CausalTransformerBlock(nn.Module):
    """Single transformer block with causal (autoregressive) self-attention."""

    def __init__(self, embed_dim: int, num_heads: int, mlp_ratio: float, dropout: float):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True,
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor, causal_mask: torch.Tensor) -> torch.Tensor:
        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed, attn_mask=causal_mask)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class CausalVisionTransformer(nn.Module):
    """Vision Transformer with causal masked self-attention for image classification.

    Patches are processed in raster-scan order (left-to-right, top-to-bottom).
    The causal mask ensures each patch only attends to itself and all preceding
    patches, mimicking an autoregressive decoder over image patches.
    The [CLS] token is prepended; its final representation drives classification.
    """

    def __init__(
        self,
        img_size: int    = 224,
        patch_size: int  = 16,
        in_channels: int = 3,
        num_classes: int = 2,
        embed_dim: int   = 256,
        num_heads: int   = 8,
        num_layers: int  = 4,
        mlp_ratio: float = 4.0,
        dropout: float   = 0.1,
    ):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches
        seq_len = num_patches + 1  # +1 for [CLS]

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, seq_len, embed_dim))
        self.pos_drop  = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            CausalTransformerBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, num_classes),
        )

        self._init_weights()
        causal = torch.triu(torch.full((seq_len, seq_len), float('-inf')), diagonal=1)
        self.register_buffer('causal_mask', causal)

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B   = x.size(0)
        x   = self.patch_embed(x)               # (B, N, D)
        cls = self.cls_token.expand(B, -1, -1)  # (B, 1, D)
        x   = torch.cat([cls, x], dim=1)        # (B, N+1, D)
        x   = self.pos_drop(x + self.pos_embed)
        for block in self.blocks:
            x = block(x, self.causal_mask)
        x       = self.norm(x)
        cls_out = x[:, 0]                       # (B, D)
        return self.head(cls_out)               # (B, num_classes)


# ── Instantiate & summarise ───────────────────────────────────────────────
model = CausalVisionTransformer(
    img_size=IMG_SIZE, patch_size=PATCH_SIZE,
    num_classes=NUM_CLASSES, embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS, num_layers=NUM_LAYERS,
    mlp_ratio=MLP_RATIO, dropout=DROPOUT,
).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: CausalVisionTransformer')
print(f'  Total params     : {total_params:,}')
print(f'  Trainable params : {trainable_params:,}')
print(f'  Sequence length  : {model.patch_embed.num_patches + 1} (patches + CLS)')

# Quick forward-pass sanity check
dummy = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f'  Output shape     : {tuple(out.shape)}  (batch=2, classes={NUM_CLASSES})')

## 4. Training (1 Epoch)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# Cosine LR scheduler over the single epoch
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=len(train_loader), eta_min=LR / 10
)


def train_one_epoch(model, loader, criterion, optimizer, scheduler, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total   = 0
    pbar = tqdm(loader, desc='Training', leave=True)
    for imgs, labels in pbar:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item() * imgs.size(0)
        preds    = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)

        pbar.set_postfix(loss=f'{loss.item():.4f}',
                         acc=f'{correct/total*100:.1f}%')

    epoch_loss = running_loss / total
    epoch_acc  = correct / total
    return epoch_loss, epoch_acc


print(f'Starting training for {NUM_EPOCHS} epoch(s) on {DEVICE}...')
train_loss, train_acc = train_one_epoch(
    model, train_loader, criterion, optimizer, scheduler, DEVICE
)
print(f'\nTrain  loss : {train_loss:.4f}')
print(f'Train  acc  : {train_acc*100:.2f}%')

## 5. Inference on Test Set

In [ ]:
def run_inference(model, loader, device):
    model.eval()
    all_preds  = []
    all_labels = []
    all_probs  = []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc='Inference', leave=True):
            imgs = imgs.to(device, non_blocking=True)
            logits = model(imgs)
            probs  = torch.softmax(logits, dim=1)
            preds  = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())  # prob of class 1
    return all_labels, all_preds, all_probs


y_true, y_pred, y_prob = run_inference(model, test_loader, DEVICE)
print(f'Inference complete. Evaluated {len(y_true):,} samples.')

## 6. Classification Metrics

In [ ]:
acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='binary', zero_division=0)
rec  = recall_score(y_true, y_pred, average='binary', zero_division=0)
f1   = f1_score(y_true, y_pred, average='binary', zero_division=0)
cm   = confusion_matrix(y_true, y_pred)

print('=' * 45)
print('  Classification Metrics (Test Set)')
print('=' * 45)
print(f'  Accuracy  : {acc*100:.2f}%')
print(f'  Precision : {prec*100:.2f}%')
print(f'  Recall    : {rec*100:.2f}%')
print(f'  F1-Score  : {f1*100:.2f}%')
print()
print(classification_report(
    y_true, y_pred,
    target_names=[CLASS_NAMES[0], CLASS_NAMES[1]],
    zero_division=0,
))

In [ ]:
import itertools

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)
tick_marks = range(NUM_CLASSES)
ax.set_xticks(tick_marks)
ax.set_yticks(tick_marks)
ax.set_xticklabels([CLASS_NAMES[i] for i in range(NUM_CLASSES)], rotation=15)
ax.set_yticklabels([CLASS_NAMES[i] for i in range(NUM_CLASSES)])

thresh = cm.max() / 2.0
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, f'{cm[i, j]:,}',
            ha='center', va='center',
            color='white' if cm[i, j] > thresh else 'black')

ax.set_ylabel('True label')
ax.set_xlabel('Predicted label')
ax.set_title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.show()

In [ ]:
y_true_arr = np.array(y_true)
y_prob_arr = np.array(y_prob)

fig, ax = plt.subplots(figsize=(7, 4))
for cls_id, color, label in [(0, '#4C72B0', CLASS_NAMES[0]),
                              (1, '#DD8452', CLASS_NAMES[1])]:
    mask = y_true_arr == cls_id
    ax.hist(y_prob_arr[mask], bins=40, alpha=0.6, color=color, label=label)
ax.axvline(0.5, color='red', linestyle='--', linewidth=1.2, label='threshold=0.5')
ax.set_xlabel('Predicted probability of "With A-H"')
ax.set_ylabel('Count')
ax.set_title('Score Distribution by True Class')
ax.legend()
plt.tight_layout()
plt.show()